# Data loading, featurization & round-trip

Loads QM9 and ZINC-250k from their **external canonical sources** (GDB-9 SDF / chemical_vae csv),
featurizes to dense `(X, E)` tensors via the `dataset` package, and reports how much each drops.

- **QM9** — element-only atom types `{C,N,O,F}`; round-trip *report-only*.
- **ZINC** — keeps `N+`/`O-`; round-trip filter *applied*.

## Setup (Colab)
Clone the repo and `uv`-install the package + deps. Run once per session.

In [ ]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
print("cwd:", os.getcwd())

## 1 · Load QM9 (GDB-9 SDF, element-only)
First call downloads + parses the SDF (~1 min); cached after. `LIMIT` caps molecules for speed.

In [ ]:
from dataset import load_qm9, load_zinc, MoleculeDataset, collate_dense

LIMIT = 5000   # set None for the full datasets

qm9 = load_qm9(local_dir="data", targets=("mu", "homo"), limit=LIMIT)
print("QM9 :", len(qm9["X"]), "molecules | y", qm9["y"].shape, "| targets", qm9["targets"])
print("  stats:", qm9["stats"])

## 2 · Load ZINC-250k (chemical_vae csv, keep N+/O-)

In [ ]:
zinc = load_zinc(local_dir="data", targets=("logP", "qed"), limit=LIMIT)
print("ZINC:", len(zinc["X"]), "molecules | y", zinc["y"].shape, "| targets", zinc["targets"])
print("  stats:", zinc["stats"])

## 3 · Featurizer round-trip checks
`smiles -> (X, E) -> smiles` should reproduce the canonical molecule.

In [ ]:
from rdkit import Chem
from dataset import smiles_to_tensor, tensor_to_mol, QM9_ATOMS, ZINC_ATOMS

def canon(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)

def check(s, vocab, charge_aware):
    X, E = smiles_to_tensor(s, atom_vocab=vocab, charge_aware=charge_aware)
    m = tensor_to_mol(X, E, atom_vocab=vocab)
    got = Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None
    print(f"  {s:22s} -> {str(got):22s} match={m is not None and got == canon(s)}")

print("QM9 (element-only):")
for s in ["CCO", "CC(=O)O", "c1ccncc1", "N#Cc1ccccc1"]:
    check(s, QM9_ATOMS, charge_aware=False)
print("ZINC (charge-aware):")
for s in ["CC[N+](C)(C)C", "CC(=O)[O-]", "Clc1ccccc1", "c1ccc2ccccc2c1"]:
    check(s, ZINC_ATOMS, charge_aware=True)

## 4 · Round-trip drop summary
`kept` = round-trips exactly; `kept_no_roundtrip` = featurized but not gated (QM9);
`drop_*` = excluded (vocab / round-trip / parse).

In [ ]:
def show(name, stats):
    n = sum(stats.values())
    kept = stats.get("kept", 0) + stats.get("kept_no_roundtrip", 0)
    print(f"{name}: {kept:,}/{n:,} usable ({kept/n:.2%})")
    for k, v in sorted(stats.items()):
        print(f"    {k:20s} {v:>6,} ({v/n:.2%})")

show("QM9 ", qm9["stats"])
print()
show("ZINC", zinc["stats"])

## 5 · Dataset + padded batch
`MoleculeDataset` + `collate_dense` pad variable-N graphs and emit a node mask — training-ready.

In [ ]:
from torch.utils.data import DataLoader

ds = MoleculeDataset.from_loader(zinc)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate_dense)
batch = next(iter(loader))
print({k: tuple(v.shape) for k, v in batch.items()})